# 1.Import Libraries

In [1]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from facenet_pytorch import MTCNN

# 2.Check GPU

In [2]:
device="cuda" if torch.cuda.is_available() else "cpu"
print("Using device:",device)

Using device: cuda


# 3.Paths

In [10]:
frame_path="/mnt/d/deepfake/tf-gpu-env/complete_project/frames"
output_path="/mnt/d/deepfake/tf-gpu-env/complete_project/faces"

real_path=os.path.join(output_path,"real")
fake_path=os.path.join(output_path,"fake")

os.makedirs(real_path,exist_ok=True)
os.makedirs(fake_path,exist_ok=True)

# 4.MTCNN

In [4]:
mtcnn=MTCNN(keep_all=True,device=device)

# 5.Sharpen Edges

In [12]:
mtcnn=MTCNN(
    keep_all=True,
    device=device,
    thresholds=[0.7,0.8,0.9]
)

# 6.Expand Face Box

In [6]:
def crop_face(frame,box,scale=1.4):

    h,w,_=frame.shape
    x1,y1,x2,y2=box

    bw=x2-x1
    bh=y2-y1

    cx=x1+bw/2
    cy=y1+bh/2

    new_w=bw*scale
    new_h=bh*scale

    x1=int(max(cx-new_w/2,0))
    y1=int(max(cy-new_h/2,0))
    x2=int(min(cx+new_w/2,w))
    y2=int(min(cy+new_h/2,h))

    face=frame[y1:y2,x1:x2]

    return face

# 7.Extract Faces

In [13]:
def extract_faces(frame_file,label):
    frame=cv2.imread(frame_file)
    if frame is None:
        return
    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    boxes,probs=mtcnn.detect(rgb)
    if boxes is None:
        return
    frame_name=os.path.basename(frame_file).split(".")[0]
    count=0
    for i,box in enumerate(boxes):
        if probs[i] is None or probs[i]<0.90:
            continue
        x1,y1,x2,y2=map(int,box)
        width=x2-x1
        height=y2-y1
        if width<80 or height<80:
            continue
        ratio=width/height
        if ratio<0.6 or ratio>1.5:
            continue
        face=crop_face(frame,[x1,y1,x2,y2])
        face=cv2.resize(face,(224,224),interpolation=cv2.INTER_CUBIC)
        face=sharpen(face)
        save_name=f"{frame_name}_{count}.jpg"
        save_path=os.path.join(output_path,label,save_name)
        cv2.imwrite(save_path,face)
        count+=1

# 8.Process Frames

In [14]:
for label in ["real","fake"]:
    folder=os.path.join(frame_path,label)
    frames=os.listdir(folder)
    print(f"Processing {label} frames...")

    for frame in tqdm(frames):
        frame_file=os.path.join(folder,frame)
        extract_faces(frame_file,label)

Processing real frames...


100%|███████████████████████████████████████████████████████████████████████████| 27423/27423 [1:41:04<00:00,  4.52it/s]


Processing fake frames...


100%|█████████████████████████████████████████████████████████████████████████████| 19698/19698 [59:47<00:00,  5.49it/s]


# 9.Images in folder

In [17]:
real_faces  = len(os.listdir("/mnt/d/deepfake/tf-gpu-env/complete_project/faces/real"))
fake_faces  = len(os.listdir("/mnt/d/deepfake/tf-gpu-env/complete_project/faces/fake"))

print("Real faces:", real_faces)
print("Fake faces:", fake_faces)
print("Total faces:", real_faces + fake_faces)

Real faces: 28226
Fake faces: 21299
Total faces: 49525
